# Week 1 Day 5 Lab 1 - Harness Engineering

## The Claude Agent SDK works the queue on its own

The Claude Agent SDK is the harness behind Claude Code, packaged as a Python library. It brings the agent loop, tools to read and write files, a task list and subagents, all ready made.

We give it a folder with five claims from the queue, the policy wording and the claims manual, and ask it to decide every claim. We don't write any tools. Then we check its decisions against the handlers' answers.

## 4 parts:

PART 1: The claims desk

PART 2: Setting up the harness

PART 3: Working the queue

PART 4: How it did

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab uses Claude Sonnet 5.5 through Anthropic's API, and a run on five claims costs about 40 cents.
            You'll need an Anthropic API key with some credit, from <a href="https://console.anthropic.com">console.anthropic.com</a>, as <code>ANTHROPIC_API_KEY</code> in your <code>.env</code> file.<br/><br/>
            If you'd rather not spend anything, please watch this one. Lab 2 in <a href="week1day5_lab2.ipynb">week1day5_lab2.ipynb</a> does the same thing with Deep Agents on OpenRouter, and it has a free option.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
import shutil
import time
from pathlib import Path

import pandas as pd
from claude_agent_sdk import AgentDefinition, AssistantMessage, ClaudeAgentOptions, ResultMessage, TextBlock, ToolUseBlock, query
from dotenv import load_dotenv
from IPython.display import Markdown, display

In [7]:
load_dotenv(override=True)

if os.getenv("ANTHROPIC_API_KEY"):
    print("Anthropic API key found")
else:
    print("No Anthropic API key found. Add ANTHROPIC_API_KEY to the .env file")

Anthropic API key found


## Part 1: The claims desk

The seed data is in the repo's `data` folder, as on the other days. We use the five claims marked for the tiny version of the queue, which include one that needs the claims manual.

In [3]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [4]:
claims, labels = get_sample_data(tiny=True)

for claim in claims:
    print(claim["claim_id"], claim["summary"])

CLM-1007 Phone dropped in the hotel shower
CLM-1015 Flew home from Paris on arrival after leaving a child at home
CLM-1017 Backpack snatched by a scooter rider in Naples
CLM-1020 Prescription sunglasses stolen by a monkey at Uluwatu Temple
CLM-1021 Tote bag snatched from her shoulder in Buenos Aires


The agent works in a folder of its own, which we'll call the claims desk. It gets one file for each claim in the queue, a file for every policy and every customer, and the policy wording, the claims manual and the company's limits.

It never sees the handlers' decisions, which stay in `data/evals`. We keep those to score the agent at the end.

In [5]:
DESK = Path("lab1_desk")


def make_desk(claims):
    """Build a fresh claims desk with the claims to decide and everything needed to decide them."""
    shutil.rmtree(DESK, ignore_errors=True)
    for folder in ["claims", "policies", "customers", "decisions"]:
        (DESK / folder).mkdir(parents=True)
    for claim in claims:
        (DESK / "claims" / f"{claim['claim_id']}.json").write_text(json.dumps(claim, indent=2), encoding="utf-8")
    for policy in load("policies.json"):
        (DESK / "policies" / f"{policy['policy_id']}.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
    for customer in load("customers.json"):
        (DESK / "customers" / f"{customer['customer_id']}.json").write_text(json.dumps(customer, indent=2), encoding="utf-8")
    for name in ["policy_wording.md", "claims_manual.md", "company.json"]:
        shutil.copy(DATA / name, DESK / name)


make_desk(claims)
sorted(path.name for path in DESK.iterdir())

['claims',
 'claims_manual.md',
 'company.json',
 'customers',
 'decisions',
 'policies',
 'policy_wording.md']

In [9]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance, working through the queue of new claims on your own.

Everything you need is in this folder:
- claims/ has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named by their ids
- policy_wording.md is the policy wording and claims_manual.md is the claims handling manual
- company.json has the company's limits

Decide every claim in claims/. Keep a task list with one task for each claim, and give each claim to the claim-handler agent, which decides it and writes its decision file.
When every claim has a decision, write handover.md at the top of this folder: a short note for the handler on what was decided and anything they should know.
"""

HANDLER_INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance. Decide the one claim you are given.

Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md for the rules that apply.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, for one of the reasons the manual gives.

Write the decision to decisions/<claim id>.json with the fields claim_id, decision, payout (in dollars, 0 unless you approve) and rationale (2-3 sentences to explain the decision along with the relevant section of the wording or the manual).
Then reply with one line: the claim id, the decision and the payout.
"""

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">The options</h2>
            <span style="color:#118AB2;"><code>tools</code> picks the built-in tools the agent can use: reading, searching and writing files, the task list (<code>TaskCreate</code>, <code>TaskUpdate</code>, <code>TaskList</code>, <code>TaskGet</code>) and <code>Task</code>, which hands work to a subagent. We leave out the shell and the web.<br/><br/>
            <code>agents</code> defines our subagent. <code>system_prompt</code> starts from Claude Code's own system prompt and adds our instructions.<br/><br/>
            <code>setting_sources=[]</code> keeps your own Claude Code settings out of the run. In <code>env</code>, <code>CLAUDE_CODE_DISABLE_AUTO_MEMORY</code> turns off the notes Claude Code keeps between sessions, so every run starts fresh, and <code>ENABLE_CLAUDEAI_MCP_SERVERS</code> leaves out any connectors from your claude.ai account.<br/><br/>
            <code>max_budget_usd</code> stops the run if it ever costs more than a dollar.
            </span>
        </td>
    </tr>
</table>

In [10]:
TOOLS = ["Read", "Glob", "Grep", "Write", "Task", "TaskCreate", "TaskUpdate", "TaskList", "TaskGet"]

claim_handler = AgentDefinition(
    description="Decides one claim and writes its decision file. Give it the claim id.",
    prompt=HANDLER_INSTRUCTIONS,
    tools=["Read", "Glob", "Grep", "Write"],
)

options = ClaudeAgentOptions(
    model="claude-sonnet-5-5",
    effort="medium",
    cwd=str(DESK),
    system_prompt={"type": "preset", "preset": "claude_code", "append": INSTRUCTIONS},
    tools=TOOLS,
    allowed_tools=TOOLS,
    agents={"claim-handler": claim_handler},
    permission_mode="dontAsk",
    setting_sources=[],
    env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1", "ENABLE_CLAUDEAI_MCP_SERVERS": "false"},
    max_budget_usd=1.00,
)

### One adjustment for Windows

The Claude Agent SDK runs Claude Code as a separate program. Inside a Jupyter notebook on Windows, the notebook's event loop can't start other programs, so `query` fails with `CLIConnectionError: Failed to start Claude Code`. The cell below gives `query` an event loop of its own in a background thread, which lets the cells below use it exactly as the SDK documents it. On Mac and Linux it does nothing at all, so run it either way and carry on.

In [11]:
import sys

if sys.platform == "win32":
    import asyncio
    import threading

    import claude_agent_sdk

    async def query(*, prompt, options):
        """The SDK's query, run on an event loop of its own in a thread, passing each message back here."""
        loop = asyncio.get_running_loop()
        messages = asyncio.Queue()

        async def run():
            try:
                async for message in claude_agent_sdk.query(prompt=prompt, options=options):
                    loop.call_soon_threadsafe(messages.put_nowait, message)
            except Exception as error:
                loop.call_soon_threadsafe(messages.put_nowait, error)
            loop.call_soon_threadsafe(messages.put_nowait, None)

        threading.Thread(target=asyncio.run, args=(run(),), kwargs={"loop_factory": asyncio.ProactorEventLoop}).start()
        while (message := await messages.get()) is not None:
            if isinstance(message, Exception):
                raise message
            yield message

    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

Applied the Windows adjustment


## Part 3: Working the queue

As the agent works we print each step. The subagent's steps are indented.

In [12]:
def describe(tool_call):
    """A short line about one tool call."""
    args = tool_call.input
    if "file_path" in args:
        return f"{tool_call.name} {Path(args['file_path']).name}"
    if tool_call.name in ("Agent", "Task"):
        return f"{tool_call.name}: {args.get('description')}"
    if tool_call.name == "TaskCreate":
        return f"TaskCreate: {args.get('subject')}"
    if tool_call.name == "TaskUpdate":
        return f"TaskUpdate: task {args.get('taskId')} {args.get('status', '')}"
    if "pattern" in args:
        return f"{tool_call.name} {args['pattern']}"
    return tool_call.name

In [13]:
result = None
start = time.time()

async for message in query(prompt="Please work the queue.", options=options):
    if isinstance(message, AssistantMessage):
        indent = "      " if message.parent_tool_use_id else ""
        for block in message.content:
            if isinstance(block, ToolUseBlock):
                print(indent + describe(block))
            elif isinstance(block, TextBlock) and block.text.strip():
                print(indent + block.text.strip())
    elif isinstance(message, ResultMessage):
        result = message

print(f"\nFinished in {time.time() - start:.0f} seconds for ${result.total_cost_usd:.2f}")

Glob claims/*
Read claims_manual.md
Read company.json
TaskCreate: Decide CLM-1007
TaskCreate: Decide CLM-1015
TaskCreate: Decide CLM-1017
TaskCreate: Decide CLM-1020
TaskCreate: Decide CLM-1021
Agent: Decide CLM-1007
Agent: Decide CLM-1015
Agent: Decide CLM-1017
Agent: Decide CLM-1020
      Glob **/CLM-1007*
Agent: Decide CLM-1021
      Glob **/CLM-1015*
      Glob *.md
      Glob *.md
      Glob **/CLM-1017*
      Glob *.md
All five claims are with claim-handler agents, which are running in the background. I'll wait for them to finish before writing the handover.
      Read CLM-1007.json
      Read CLM-1015.json
      Glob **/CLM-1021*
      Glob **/*.md
      Read policy_wording.md
      Read policy_wording.md
      Read claims_manual.md
      Read claims_manual.md
      Read CLM-1017.json
      Glob **/CLM-1020*
      Glob *.md
      Read policy_wording.md
      Read claims_manual.md
      Read CLM-1021.json
      Read policy_wording.md
      Read POL-2015.json
      Read claims_man

## Part 4: How it did

Each decision is a file on the claims desk. We compare them with the handlers' decisions, using the same test as on Day 2: the decision has to match and the payout has to be within a dollar.

In [14]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer.get("decision") == label["decision"] and abs(answer.get("payout", 0) - label["payout"]) <= 1


rows = []
for claim_id, label in labels.items():
    path = DESK / "decisions" / f"{claim_id}.json"
    answer = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"decision": "missing", "payout": 0}
    rows.append({
        "claim": claim_id,
        "decision": answer.get("decision"),
        "payout": answer.get("payout"),
        "handler's decision": label["decision"],
        "handler's payout": label["payout"],
        "right": is_right(answer, label),
    })

results = pd.DataFrame(rows)
print(f"{results['right'].sum()} of {len(results)} right")
results

5 of 5 right


,claim,decision,payout,handler's decision,handler's payout,right
0,CLM-1007,approve,50,approve,50.0,True
1,CLM-1015,decline,0,decline,0.0,True
2,CLM-1017,refer,0,refer,0.0,True
3,CLM-1020,approve,240,approve,240.0,True
4,CLM-1021,approve,825,approve,825.0,True


And here's the note it left for the handler.

In [15]:
display(Markdown((DESK / "handover.md").read_text(encoding="utf-8")))

# Handover: claims queue

All five claims in `claims/` have a decision file in `decisions/`.

| Claim | Decision | Payout | Summary |
|---|---|---|---|
| CLM-1007 | Approve | $50 | Phone damaged accidentally. The Economy $150 per-item limit applies, less the $100 deductible. No Electronics Plus add-on. |
| CLM-1015 | Decline | $0 | Trip cut short because a child was left at home by mistake. This is not one of the covered interruption reasons (wording s2). |
| CLM-1017 | Refer | $0 | Snatch theft. Two documents are missing: the Carabinieri police report and proof of value for the $130 sunglasses. A 30-day document request applies. The claim looks payable once they arrive. |
| CLM-1020 | Approve | $240 | Sunglasses taken by a macaque. Treated as accidental loss under the wild-animal note, so no police report is needed. The licensed guide's letter and the replacement receipt are on file. First plan, no deductible. |
| CLM-1021 | Approve | $825 | Theft from the person. Electronics Plus applies only to the e-reader. The headphones and console are capped at $300 each. Total of $875, less the $50 deductible. |

## Things to know

- **Sign-off:** no payout is above $1,000, so none needs sign-off. CLM-1021 is the highest at $825.
- **Repeat claims:** the agents reported no earlier claims for these customers (CLM-1007, CLM-1015, CLM-1020, CLM-1021). I did not check the customer files myself.
- **CLM-1017:** the claim stays open while the customer has 30 days to send the two documents. Someone needs to follow up.
- **Decision records:** the decision files hold the decision, payout, reason and section relied on. They do not hold the manual version (2026.2), who decided or when, which manual s13–14 asks for. This may need adding.
- **Spot checks:** I read the five decision files but did not re-work the calculations or re-read the claims, policies or customer records myself.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Build or buy</h2>
            <span style="color:#118AB2;">A ready-made harness like this one gets you a capable agent with very little code. The trade-offs are that it runs only Claude models, and it starts a Claude Code program for every run.<br/><br/>
            Weighing these trade-offs for a customer is a big part of an AI Solutions Architect's job.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Look back through the steps and find where the agent used its task list and where it handed work to the subagent.<br/><br/>
            Then try one change and see what it does to accuracy and cost: a different effort level, a different model such as <code>claude-haiku-4-5</code>, or a change to the instructions.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>